In [14]:
import json
from rapidfuzz import fuzz
from itertools import combinations
from collections import Counter
from sentence_transformers import SentenceTransformer, util


In [26]:
input_path = "ontology_output/ontology_fabi_experimet.json"
output_path = "ontology_output/ontology_fabi_experimet_synonyms_replaced.json"

In [15]:
def get_common_terms(data, field_name, top_n=20):
    counter = Counter()
    for cluster in data:
        terms = cluster.get("ontology", {}).get(field_name, [])
        counter.update(terms)
    return counter.most_common(top_n), list(counter.keys())

def find_similar_by_string(terms, threshold=80):
    similar_pairs = []
    for a, b in combinations(terms, 2):
        score = fuzz.ratio(a, b)
        if score >= threshold:
            similar_pairs.append((a, b, score))
    similar_pairs.sort(key=lambda x: -x[2])
    return similar_pairs

def find_similar_by_embedding(terms, model, threshold=0.8):
    embeddings = model.encode(terms, convert_to_tensor=True)
    similar_pairs = []
    for i, j in combinations(range(len(terms)), 2):
        score = util.cos_sim(embeddings[i], embeddings[j]).item()
        if score >= threshold:
            similar_pairs.append((terms[i], terms[j], score))
    similar_pairs.sort(key=lambda x: -x[2])
    return similar_pairs

In [27]:
# Load file 
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [17]:
model = SentenceTransformer('distiluse-base-multilingual-cased-v1')
results = {}

for category in ["Klassen", "Eigenschaften"]:
    common_terms, all_terms = get_common_terms(data, category)
    
    string_similar = find_similar_by_string(all_terms)
    embedding_similar = find_similar_by_embedding(all_terms, model)
    
    results[category] = {
        "common_terms": common_terms,
        "string_similar": string_similar,
        "embedding_similar": embedding_similar,
    }

    # Save string similarity
    with open(f"top_similar_{category.lower()}_string.json", "w", encoding="utf-8") as f_out:
        json.dump(
            [{"term_1": a, "term_2": b, "levenshtein_score": score} for a, b, score in string_similar],
            f_out, indent=2, ensure_ascii=False
        )

    # Save embedding similarity
    with open(f"top_similar_{category.lower()}_embedding.json", "w", encoding="utf-8") as f_out:
        json.dump(
            [{"term_1": a, "term_2": b, "similarity": score} for a, b, score in embedding_similar],
            f_out, indent=2, ensure_ascii=False
        )

results["Klassen"]["common_terms"][:10], results["Eigenschaften"]["common_terms"][:10]

([('Rechtsgrundlage', 24),
  ('Rechtsfolge', 21),
  ('Vertrag', 15),
  ('Verjährung', 14),
  ('Anspruch', 13),
  ('Mangel', 13),
  ('Schadensersatzanspruch', 12),
  ('Schaden', 10),
  ('Mietvertrag', 9),
  ('Verpflichtung', 9)],
 [('erfüllen', 30),
  ('gilt für', 23),
  ('beinhaltet', 22),
  ('betrifft', 21),
  ('verpflichtet', 21),
  ('erfüllt', 17),
  ('hat', 17),
  ('geltend machen', 17),
  ('gilt', 15),
  ('tritt ein', 14)])

Some issues occured, because it found sting similarities in oposites, like in Verletzer <-> Verletzter, but it gives a good guidence for finding duplicates like in Netznutzungsentgelte <-> Netznutzungsentgelt.

In [ ]:
duplicate_class_sets = [
    {"Kläger", "Klägerin"},
    {"Netznutzungsentgelt", "Netznutzungsentgelte"},
    {"Angeklagter", "Angeklagte"},
    {"Verurteilter", "Verurteilte"},
    {"Geräuschemission", "Geräuschemissionen"},
    {"Mangelbeseitigung", "Mängelbeseitigung", "Mangelbeseitigungen", "Mängelbeseitigungen"},
    {"Mangel", "Mängel"},
    {"Geschäft", "Geschäfte"},
    {"Beklagter", "Beklagte"},
    {"Geselschafter", "Gesellschafterin"},
    {"Voraussetzung", "Voraussetzungen"},
    {"Rechtsfolge", "Rechtsfolgen"},
    {"Beklagter", "Beklagte"},
    {"Gemeinschuldner", "Gemeinschuldnerin"},
    {"Schadensersatzanspruch", "Schadensersatzansprüche", "Schadensanspruch", "Schadensansprüche"},
    {"Drittschuldner", "Drittschuldnerin"},
    {"Recht", "Rechte"},
    {"Vermieter", "Vermieterin"},
    {"Bedingungen", "Bedingung"},
    {"Mangelanspruch", "Mangelansprüche", "Mängelanspruch", "Mängeransprüche"},
    {"Schadensersatzpflicht", "Schadensersatzverpflichtung"},
    {"Schadensersatz", "Schadensersatzleistung"},
    {"Gebühren", "Gebühr"},
    {"Geschäftsführer", "Geschäftsführerin", "Geschäftsführung"},
    {"Umstände", "Umstand"},
    {"Anklageschrift", "Klageschrift"}, # Eigentlich Zivil vs Strafprozess, aber ähnlich
    {"Richter", "Richteramt"}, # Vielleicht
    {"Schlichtungsverfahren", "Schlichtungsversuch"},
    {"Kündigungsklausel", "Hinauskündigungsklausel"},
    {"Verrechnung", "Aufrechnung"}
]

duplicate_property_sets = [
    {"hat_Anspruch_auf", "hat_Anpruch_auf"},
    {"erfüllen", "erfüllt"},
    {"verletzen", "verletzt"},
    {"hat Bedeutung", "hatBedeutung"},
    {"spieltRolle", "spielt Rolle", "spielt_Rolle_in", "spielt_rolle_in", "spielt_Rolle_bei", "spielt eine Rolle",	"spielt eine Rolle bei"},
    {"hatKenntnis", "hat Kenntnis"},
    {"führt zu", "führtzu"},
    {"hat Rechtsfolgen", "hatRechtsfolge"},
    {"sicherstellen", "sicherzustellen"},
    {"geltend",	"gelten"},
    {"entscheiden",	"entscheidet"},
    {"istVerpflichtet", "ist verpflichtet"},
    {"verhindern", "verhindert"},
    {"betrachten", "betrachtet"},
    {"verweigern",	"verweigert"},
    {"beeinträchtigt",	"beeinträchtigen"},
    {"berücksichtigen",	"berücksichtigt"},
    {"geltend machen",	"geltendMachen"},
    {"beanspruchen können",	"beanspruchen kann"},
    {"interpretieren",	"interpretiert"},
    {"begründen", "begründet"},
    {"ermitteln", "ermittelt"},
    {"erforderlich für", "ist erforderlich für"},
    {"hat Auswirkungen auf", "hat Auswirkungen"},
    {"muss erfüllt sein", "müssen erfüllt sein"},
    {"istUnzulässig", "ist unzulässig"},
    {"verstößtGegen", "verstößt gegen"},
    {"ist erforderlich", "ist erforderlich für"}
    #45




    #"hat eine Bedeutung", "hat eine Bedeutung für"
    #"kann geltend gemacht werden", "kann geltend gemacht werden"
    #"eingeräumt",	"einräumt"
    #"auszulegen",	"auslegen"






    {"verursachen", "verursacht"},
    {"beeinträchtigen", "beeinträchtigt"},
    {"verletzt", "verletzen"},
    {"verursacht", "verursachen"},
    {"beeinträchtigt", "beeinträchtigen"},
]

# Aus duplicate_sets eine Lookup-Tabelle bauen
synonym_class_map = {}

for synonym_group in duplicate_class_sets:
    representative = sorted(synonym_group)[0]  # Alphabetisch erster Eintrag als Repräsentant
    for variant in synonym_group:
        synonym_class_map[variant] = representative

synonym_property_map = {}

for synonym_group in duplicate_property_sets:
    representative = sorted(synonym_group)[0]  # Alphabetisch erster Eintrag als Repräsentant
    for variant in synonym_group:
        synonym_property_map[variant] = representative

In [28]:
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)
    
class_counter = Counter()
property_counter = Counter()

for cluster in data:
    ontology = cluster.get("ontology", {})
    klassen = ontology.get("Klassen", [])
    eigenschaften = ontology.get("Eigenschaften", [])

    # Normalisieren
    ontology["Klassen"] = [synonym_class_map.get(k, k) for k in klassen]
    ontology["Eigenschaften"] = [synonym_property_map.get(e, e) for e in eigenschaften]

    class_counter.update(klassen)
    property_counter.update(eigenschaften)

most_common_classes = class_counter.most_common(20)
most_common_properties = property_counter.most_common(20)

print(f"Most common classes: {most_common_classes}")
print(f"Most common properties: {most_common_properties}")

with open(output_path, "w", encoding="utf-8") as f_out:
    json.dump(data, f_out, indent=2, ensure_ascii=False)


Most common classes: [('Rechtsgrundlage', 24), ('Rechtsfolge', 21), ('Vertrag', 15), ('Verjährung', 14), ('Anspruch', 13), ('Mangel', 13), ('Schadensersatzanspruch', 12), ('Schaden', 10), ('Mietvertrag', 9), ('Verpflichtung', 9), ('Klausel', 9), ('Kunde', 8), ('Haftung', 8), ('Mieter', 7), ('Käufer', 7), ('Verkäufer', 7), ('Kündigung', 7), ('Makler', 7), ('Vergütung', 7), ('Gesellschafter', 6)]
Most common properties: [('erfüllen', 30), ('gilt für', 23), ('beinhaltet', 22), ('betrifft', 21), ('verpflichtet', 21), ('erfüllt', 17), ('hat', 17), ('geltend machen', 17), ('gilt', 15), ('tritt ein', 14), ('tragen', 10), ('sicherstellen', 9), ('spielt eine Rolle', 9), ('spielen', 9), ('verletzen', 8), ('verletzt', 8), ('wirksam', 8), ('verlangen', 8), ('bestimmen', 8), ('basiert auf', 8)]
